# How Noisy Is a Mean Return?

Modules 04 and 05 took expected returns, volatilities and correlations as given. In practice they
are estimated from history, and the expected return is by far the hardest of the three. Ten years of
Swiss equity returns pin down the mean only to within about ±10 percentage points per year. This
notebook derives the standard error of a mean return, shows why sampling more often does not help,
and turns the result into rules of thumb for track records and backtests.

## 0. Math link

- [Sample mean](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/02_sample_mean.ipynb):
  $\bar X$ is unbiased for $\mu$ and has variance $\sigma^2/n$ for independent observations.
- [Central limit theorem](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/04_sampling_and_point_estimation/07_central_limit_theorem_continuous.ipynb):
  $\bar X$ is approximately normal for large $n$, even when the returns themselves have fat tails.
- [t distribution](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/06_confidence_intervals/07_t_distribution.ipynb):
  the interval to use when $\sigma$ is estimated from only a few observations.
- Log returns add up over time, and volatility scales with $\sqrt{T}$ (both covered in module 00, planned).

## 1. Introduction

A fund has returned 9% per year over the last five years. A backtest shows 11% per year over twenty.
A capital-market assumption says Swiss equities return 6.5%. How much of each number is signal?

The answer depends on one quantity, the **standard error** of the mean, and on one surprising fact:
for the mean, the precision depends on **how many years** of data you have, not on how many
observations.

After this notebook you can:

- compute the standard error and a confidence interval for an annualized mean return;
- explain why daily data does not estimate the mean better than annual data over the same span, while it
  does estimate volatility better;
- compute how many years are needed to estimate a mean, or tell two means apart, to a given precision;
- check by simulation that the normal interval works for fat-tailed daily returns, and see where it fails.

## 2. Theory

### Standard error of a mean return

Let $r_1, \dots, r_n$ be independent returns with mean $\mu$ and volatility $\sigma$ (same periodicity). The
sample mean $\bar r = \frac1n\sum_t r_t$ satisfies

$$
E[\bar r] = \mu, \qquad \operatorname{Var}(\bar r) = \frac{\sigma^2}{n}, \qquad
\operatorname{SE}(\bar r) = \frac{\sigma}{\sqrt n}
$$

By the central limit theorem, $\bar r$ is approximately $N(\mu, \sigma^2/n)$, so an approximate 95% confidence interval is
$\bar r \pm 1.96\,\hat\sigma/\sqrt n$.

### Annualized: only the span counts

Take $T$ years of data sampled $m$ times per year ($m = 252$ daily, $12$ monthly, $1$ annual), so $n = mT$. Per
period, the mean is $\mu/m$ and the volatility $\sigma/\sqrt m$, where $\mu$ and $\sigma$ are annual. The annualized
estimate $\hat\mu = m\,\bar r$ has

$$
\operatorname{SE}(\hat\mu) = m\cdot\frac{\sigma/\sqrt m}{\sqrt{mT}} = \frac{\sigma}{\sqrt T}
$$

The sampling frequency $m$ cancels. With log returns the result is exact, not just approximate: log returns
add up, so

$$
\hat\mu = \frac{1}{T}\sum_{t=1}^{mT} \log r_t = \frac{\ln P_T - \ln P_0}{T}
$$

The estimate depends only on the **first and last price**. Everything in between (daily wiggles,
crashes and recoveries) cancels out. Sampling more often adds observations but no information about
the mean.

### Volatility is different

The sample variance of $n$ normal observations has $\operatorname{SE}(\hat\sigma^2) = \sigma^2\sqrt{2/(n-1)}$, so

$$
\operatorname{SE}(\hat\sigma) \approx \frac{\sigma}{\sqrt{2n}} = \frac{\sigma}{\sqrt{2mT}}
$$

Here $m$ does **not** cancel: daily data over 10 years gives $n = 2{,}520$ and a volatility estimate about
$\sqrt{252} \approx 16$ times more precise than 10 annual observations. Volatility lives in the wiggles; the mean
lives in the endpoints. (Module 06/04 develops this: means are hard, covariances are easier.)

### How many years?

Solving $\sigma/\sqrt T = s$ for $T$:

$$
T = \left(\frac{\sigma}{s}\right)^2
$$

Halving the standard error needs four times as many years. For the difference of two assets with
volatilities $\sigma_1, \sigma_2$ and correlation $\rho$, replace $\sigma$ by the volatility of the difference,
$\sqrt{\sigma_1^2 + \sigma_2^2 - 2\rho\sigma_1\sigma_2}$.

### The t-statistic and the Sharpe ratio

The t-statistic of a mean excess return is

$$
t = \frac{\hat\mu - r_f}{\sigma/\sqrt T} = \mathrm{SR}\cdot\sqrt T
$$

so the years needed for $t \ge 2$ are $T = (2/\mathrm{SR})^2$. A Sharpe ratio of 0.5 needs 16 years; 0.25 needs 64.

### When the normal interval fails

- **Few observations with estimated $\sigma$**: with $n$ annual returns, $(\bar r - \mu)/(\hat\sigma/\sqrt n)$ has a $t_{n-1}$
  distribution (for normal returns), whose quantiles are wider than 1.96. With $n = 5$, the 97.5% quantile is 2.78.
- **Dependence**: if returns are autocorrelated, $\operatorname{Var}(\bar r) \ne \sigma^2/n$ (exercise 6).
- **A changing $\mu$**: the formula assumes one true mean over the whole sample. If the mean drifts, a long
  sample estimates an average of different regimes.

## 3. Worked Examples

### Example 1: ten years of Swiss equities

Volatility $\sigma = 16\%$ per year, $T = 10$ years:

$$
\operatorname{SE} = \frac{16\%}{\sqrt{10}} = 5.06\%, \qquad \text{95\% CI: } 6.5\% \pm 1.96 \cdot 5.06\% = [-3.4\%,\ 16.4\%]
$$

A sample mean of 6.5% is consistent with a true expected return anywhere from slightly negative to
above 16% per year.

### Example 2: daily data does not help

The same 10 years as 2,520 daily returns. Daily volatility $16\%/\sqrt{252} = 1.008\%$, daily standard error
$1.008\%/\sqrt{2520} = 0.0201\%$; annualized $252 \cdot 0.0201\% = 5.06\%$. Identical.

For volatility, 10 annual returns give $\operatorname{SE}(\hat\sigma) \approx 16\%/\sqrt{20} = 3.6\%$; 2,520 daily returns give
$16\%/\sqrt{5040} = 0.23\%$.

### Example 3: how many years?

For $\operatorname{SE} = 1\%$: $T = (16/1)^2 = 256$ years. For $\operatorname{SE} = 2\%$: $T = 64$ years. For Swiss bonds ($\sigma = 5\%$) and
$\operatorname{SE} = 1\%$: $T = 25$ years. Bond means are easier to estimate because bonds are less volatile, not
because the data is better.

### Example 4: is equity better than bonds?

Swiss equities ($\mu = 6.5\%$, $\sigma = 16\%$) and Swiss bonds ($\mu = 2\%$, $\sigma = 5\%$), correlation 0.1. The difference has
volatility $\sqrt{0.0256 + 0.0025 - 2 \cdot 0.1 \cdot 0.16 \cdot 0.05} = \sqrt{0.0265} = 16.3\%$. A 4.5-point difference is two standard
errors when $16.3\%/\sqrt T = 2.25\%$, i.e. $T = 52$ years.

### Example 5: a Sharpe ratio of 0.375

Swiss equities with $r_f = 0.5\%$: $\mathrm{SR} = (6.5\% - 0.5\%)/16\% = 0.375$. With 10 years, $t = 0.375\sqrt{10} = 1.19$. For $t = 2$:
$T = (2/0.375)^2 = 28.4$ years.

## 4. Finance Application

### Where it matters

- **Fund selection.** A three-year track record of a fund with 15% volatility has a standard error of
  $15\%/\sqrt3 = 8.7\%$ per year. Ranking funds by three-year returns ranks them mostly by luck.
- **Capital-market assumptions.** Long-run expected returns are not read off historical averages
  alone; they combine history with economic models (yields, dividends, risk premia), because history
  cannot deliver the precision.
- **Backtests.** A strategy with an excess return of 3% and 10% tracking error ($\mathrm{SR} = 0.3$) needs about 44 years
  for $t = 2$. A 10-year backtest showing it is evidence of very little, and that is before accounting for the many
  strategies tried (06/03).
- **Optimization.** Mean-variance weights react strongly to expected returns (04/04). Feeding them
  estimates with a ±10-point confidence interval produces extreme portfolios (08/01).

### The usual misreadings

- "The 10-year average return was 8%, so the expected return is 8%." It is an estimate with a
  standard error of about $\sigma/\sqrt{10}$, here 5 points for an equity fund.
- "Use daily data to get more observations." For the mean it changes nothing; the estimate of the
  annual log mean depends only on the start and end prices. It does sharpen the volatility estimate.
- "A 95% interval of [−3%, 16%] means the true mean is in it with 95% probability." The interval is
  random, the mean is fixed: 95% of intervals built this way contain the true mean (06/02 and the
  [confidence vs probability](https://github.com/dnlwrthstr/LearningProbability/blob/main/coursera/06_confidence_intervals/06_confidence_vs_probability.ipynb)
  notebook).
- "Fat tails make the interval useless." For the mean of many daily returns the CLT works well
  (Python section). Fat tails hurt more for volatility and tail-risk estimates.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
trading_days = 252

### Simulating many 10-year histories

Simulated daily **log** returns for Swiss equities: annual volatility $\sigma = 16\%$ and annual log mean
$m = \mu - \sigma^2/2 = 6.5\% - 1.28\% = 5.22\%$ (the log mean that matches a 6.5% expected simple return for lognormal prices).
Two versions: normal, and Student-t with 4 degrees of freedom scaled to the same volatility (fat tails, as in 05/02).
Each row is one possible 10-year history.

In [ ]:
sigma = 0.16                       # annual volatility
m_log = 0.065 - sigma**2 / 2       # annual log mean, decimal
years = 10
n_days = years * trading_days
n_paths = 5_000

mu_d = m_log / trading_days
sigma_d = sigma / np.sqrt(trading_days)

def daily_log_returns(n_paths, n_days, rng, df=None):
    # daily log returns, shape (n_paths, n_days); df=None for normal, else Student-t scaled to unit variance
    if df is None:
        z = rng.standard_normal((n_paths, n_days))
    else:
        z = rng.standard_t(df, size=(n_paths, n_days)) / np.sqrt(df / (df - 2))
    return mu_d + sigma_d * z

R_norm = daily_log_returns(n_paths, n_days, rng)
R_t4 = daily_log_returns(n_paths, n_days, rng, df=4)

mean_norm = R_norm.mean(axis=1) * trading_days      # annualized mean log return per history
mean_t4 = R_t4.mean(axis=1) * trading_days

pd.DataFrame({
    "average of estimates": [mean_norm.mean(), mean_t4.mean()],
    "std of estimates (empirical SE)": [mean_norm.std(), mean_t4.std()],
    "theory sigma/sqrt(T)": [sigma / np.sqrt(years)] * 2,
    "share of histories with negative mean": [(mean_norm < 0).mean(), (mean_t4 < 0).mean()],
}, index=["normal daily", "t(4) daily"]).T.round(4)

The true annual log mean is 5.22%. Across 5,000 possible 10-year histories the estimate scatters with a
standard deviation of about 5%, as $\sigma/\sqrt T$ predicts, and roughly one history in seven shows a negative average
return. Fat-tailed daily returns change nothing here: the mean of 2,520 of them is close to normal.

### Only the endpoints matter

For one history, compute the annualized mean from daily, monthly (21-day) and annual (252-day) log returns.

In [ ]:
path = R_norm[0]
log_price = np.concatenate([[0.0], np.cumsum(path)])     # log price, starting at ln(1) = 0

daily = path
monthly = path.reshape(-1, 21).sum(axis=1)
annual = path.reshape(-1, trading_days).sum(axis=1)

print(f"daily   ({daily.size:5d} obs): {daily.mean() * trading_days:.6%}")
print(f"monthly ({monthly.size:5d} obs): {monthly.mean() * 12:.6%}")
print(f"annual  ({annual.size:5d} obs): {annual.mean():.6%}")
print(f"(ln P_T - ln P_0) / T          : {(log_price[-1] - log_price[0]) / years:.6%}")

The four numbers agree to rounding. This history happened to lose about 5% a year in log terms, although
the true log mean is +5.22%. Now the same comparison over all histories, for the mean and for the
volatility. For volatility we annualize the sample standard deviation with $\sqrt m$.

In [ ]:
def by_frequency(R):
    rows = {}
    for label, k, per_year in [("daily", 1, 252), ("monthly", 21, 12), ("annual", 252, 1)]:
        agg = R.reshape(R.shape[0], -1, k).sum(axis=2)
        rows[label] = {
            "obs per history": agg.shape[1],
            "SE of mean estimate": (agg.mean(axis=1) * per_year).std(),
            "SE of vol estimate": (agg.std(axis=1, ddof=1) * np.sqrt(per_year)).std(),
            "theory SE vol sigma/sqrt(2n)": sigma / np.sqrt(2 * agg.shape[1]),
        }
    return pd.DataFrame(rows).T

freq_table = by_frequency(R_norm)
freq_table.style.format({"obs per history": "{:.0f}", "SE of mean estimate": "{:.2%}",
                         "SE of vol estimate": "{:.2%}", "theory SE vol sigma/sqrt(2n)": "{:.2%}"})

The standard error of the mean is 5.06% at every frequency. The standard error of volatility falls from about
3.7% with 10 annual returns to 0.23% with 2,520 daily ones. (With 10 annual observations the theory value
$\sigma/\sqrt{2n}$ is only a rough guide.)

### Years needed

In [ ]:
def years_needed(sigma, se):
    return (sigma / se) ** 2

def years_for_t(sharpe, t=2.0):
    return (t / sharpe) ** 2

rows = []
for name, s in [("Swiss bonds", 0.05), ("global equities", 0.17), ("Swiss equities", 0.16)]:
    rows.append({"asset": name, "vol": s, "SE 10y": s / np.sqrt(10),
                 "years for SE 2%": years_needed(s, 0.02), "years for SE 1%": years_needed(s, 0.01)})
pd.DataFrame(rows).set_index("asset").style.format(
    {"vol": "{:.0%}", "SE 10y": "{:.2%}", "years for SE 2%": "{:.0f}", "years for SE 1%": "{:.0f}"})

In [ ]:
vol_diff = np.sqrt(0.16**2 + 0.05**2 - 2 * 0.1 * 0.16 * 0.05)
print(f"vol of equity-minus-bond return: {vol_diff:.2%}")
print(f"years for the 4.5-point gap to be 2 SE: {years_needed(vol_diff, 0.045 / 2):.1f}")
for sr in [1.0, 0.5, 0.375, 0.25]:
    print(f"Sharpe {sr:.3f}: years for t = 2: {years_for_t(sr):6.1f}")

### Do 95% intervals cover 95% of the time?

For each history build the interval $\hat\mu \pm 1.96\,\hat\sigma/\sqrt T$ from its first 1, 3 or 10 years of daily data, with
$\hat\sigma$ the annualized sample volatility. Count how often it contains the true log mean.

In [ ]:
def coverage_daily(R, years_used, z=1.96):
    sub = R[:, :years_used * trading_days]
    mu_hat = sub.mean(axis=1) * trading_days
    sig_hat = sub.std(axis=1, ddof=1) * np.sqrt(trading_days)
    half = z * sig_hat / np.sqrt(years_used)
    return np.mean(np.abs(mu_hat - m_log) <= half)

pd.DataFrame({
    "normal daily": [coverage_daily(R_norm, y) for y in [1, 3, 10]],
    "t(4) daily": [coverage_daily(R_t4, y) for y in [1, 3, 10]],
}, index=["1 year", "3 years", "10 years"]).style.format("{:.1%}")

All close to 95%, fat tails included. The daily data delivers a precise $\hat\sigma$, and the CLT does the rest.

The picture changes with **few** observations. With only 5 annual returns, $\hat\sigma$ itself is noisy. The z-interval
then undercovers; the $t_4$ quantile corrects it.

In [ ]:
n_obs = 5
annual_R = R_norm[:, :n_obs * trading_days].reshape(n_paths, n_obs, trading_days).sum(axis=2)
mu_hat = annual_R.mean(axis=1)
se_hat = annual_R.std(axis=1, ddof=1) / np.sqrt(n_obs)

z_crit = stats.norm.ppf(0.975)
t_crit = stats.t.ppf(0.975, df=n_obs - 1)
print(f"critical values: z = {z_crit:.2f}, t_{n_obs - 1} = {t_crit:.2f}")
print(f"coverage with z: {np.mean(np.abs(mu_hat - m_log) <= z_crit * se_hat):.1%}")
print(f"coverage with t: {np.mean(np.abs(mu_hat - m_log) <= t_crit * se_hat):.1%}")

### One long history: how fast does the estimate settle?

Fifty years of daily returns. After $t$ years the running estimate is (log price after $t$ years) / $t$.

In [ ]:
long_years = 50
R_long = daily_log_returns(1, long_years * trading_days, rng)[0]
t_years = np.arange(1, R_long.size + 1) / trading_days
running = np.cumsum(R_long) / t_years

for y in [1, 5, 10, 20, 50]:
    i = y * trading_days - 1
    print(f"after {y:2d} years: estimate {running[i]:7.2%},  ±1.96 SE = ±{1.96 * sigma / np.sqrt(y):.2%}")

After 50 years this history estimates 1.5% against a true 5.22%, still inside the ±4.4-point band. After one year the
estimate is −16%; the band is ±31 points.

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bins = np.linspace(-0.15, 0.25, 61)
ax.hist(mean_norm, bins=bins, density=True, histtype="step", linewidth=1.5, label="normal daily returns")
ax.hist(mean_t4, bins=bins, density=True, histtype="step", linewidth=1.5, label="t(4) daily returns")
x = np.linspace(-0.15, 0.25, 400)
ax.plot(x, stats.norm.pdf(x, m_log, sigma / np.sqrt(years)), color="black", linestyle="--",
        label=r"N(m, $\sigma^2/T$)")
ax.axvline(m_log, color="black", linewidth=0.8)
ax.axvline(0, color="grey", linewidth=0.8, linestyle=":")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Annualized mean log return estimated from 10 years of daily data (5,000 histories)")
ax.set_xlabel("estimated annual mean log return")
ax.set_ylabel("density")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
T_grid = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(9, 5))
for name, s, c in [("Swiss bonds, 5% vol", 0.05, "tab:green"), ("Swiss equities, 16% vol", 0.16, "tab:red")]:
    ax.plot(T_grid, s / np.sqrt(T_grid), color=c, label=name)
for level in [0.01, 0.02]:
    ax.axhline(level, color="grey", linewidth=0.8, linestyle=":")
ax.plot(64, 0.02, "o", color="tab:red")
ax.annotate("64 years for SE 2%", (64, 0.02), textcoords="offset points", xytext=(5, 8))
ax.plot(25, 0.01, "o", color="tab:green")
ax.annotate("25 years for SE 1%", (25, 0.01), textcoords="offset points", xytext=(5, -15))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_ylim(0, 0.12)
ax.set_title(r"Standard error of the annual mean return, $\sigma/\sqrt{T}$")
ax.set_xlabel("years of data T")
ax.set_ylabel("standard error (annual)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
show = t_years >= 0.25                 # skip the first quarter, where the estimate swings wildly
band = 1.96 * sigma / np.sqrt(t_years[show])
ax.fill_between(t_years[show], m_log - band, m_log + band, color="tab:blue", alpha=0.15, label="true m ± 1.96 SE")
ax.plot(t_years[show], running[show], color="tab:blue", linewidth=1, label="running estimate, one history")
ax.axhline(m_log, color="black", linewidth=0.8, label=f"true m = {m_log:.2%}")
ax.set_ylim(-0.4, 0.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_title("Running estimate of the annual mean log return")
ax.set_xlabel("years of data")
ax.set_ylabel("annual mean log return")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
labels = ["daily", "monthly", "annual"]
x_pos = np.arange(3)
axes[0].bar(x_pos, freq_table["SE of mean estimate"], color="tab:blue")
axes[0].set_title("SE of the annual mean, 10 years of data")
axes[1].bar(x_pos, freq_table["SE of vol estimate"], color="tab:orange")
axes[1].set_title("SE of the annual volatility, 10 years of data")
for ax in axes:
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f"{l}\n(n = {int(n)})" for l, n in zip(labels, freq_table["obs per history"])])
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.1%}"))
    ax.grid(True, alpha=0.3, axis="y")
axes[1].set_ylim(axes[0].get_ylim())
plt.tight_layout()
plt.show()

## 7. Exercises

### Basic

1. Global equities have 17% volatility. Compute the standard error and the 95% confidence interval of a
   5-year average return of 7%.
2. How many years of data are needed to estimate the expected return of a portfolio with 11.1% volatility
   (the 40/30/30 portfolio of 04/01) with a standard error of 2%?
3. A fund reports an average return of 12% per year over three years with 15% volatility, and the
   marketing text says it "beat the market's 7% by 5 points a year". **Interpret** the 12% for an investor.

### Intermediate

4. Show numerically that for log returns the annualized mean from weekly data (5-day blocks) equals the
   one from daily data for the same history. Then repeat with **simple** returns, computed as
   `np.expm1(path)`: are the daily and annual estimates still equal? How large is the difference?
5. Repeat the small-sample coverage check with $n = 3$, $10$ and $30$ annual returns, for the z and t intervals.
   How far below 95% is the z-interval at each $n$? Compare with the exact z-interval coverage for normal returns,
   $P(|t_{n-1}| \le 1.96)$.

### Advanced

6. Monthly returns with autocorrelation: simulate $r_t = \mu + \phi(r_{t-1} - \mu) + \varepsilon_t$ with $\phi = 0.2$, scaled so the
   monthly volatility is $16\%/\sqrt{12}$, for 10 years and 5,000 histories. Compare the empirical standard error of
   the annualized mean with $\sigma/\sqrt T$ and with the long-run formula $\frac{\sigma}{\sqrt T}\sqrt{\frac{1+\phi}{1-\phi}}$.
7. A manager has an information ratio (active return / tracking error) of 0.5. How many years until the
   active return is significant at $t = 2$? A pension fund reviews managers after 3 years and fires those with
   negative active return. What fraction of such skilled managers is fired at the review, assuming normality?

## 8. Solutions / Checks

In [ ]:
# 1.
se1 = 0.17 / np.sqrt(5)
print(f"1. SE = {se1:.2%}, 95% CI = [{0.07 - 1.96 * se1:.1%}, {0.07 + 1.96 * se1:.1%}]")

# 2.
print(f"2. years for SE 2% at 11.1% vol: {years_needed(0.111, 0.02):.0f}")

# 3.
se3 = 0.15 / np.sqrt(3)
print(f"3. SE = {se3:.2%}, 95% CI = [{0.12 - 1.96 * se3:.1%}, {0.12 + 1.96 * se3:.1%}]")

In [ ]:
# 4.
path = R_norm[0]
weekly = path.reshape(-1, 5).sum(axis=1)
print(f"4. log, daily: {path.mean() * 252:.6%}   weekly: {weekly.mean() * (252 / 5):.6%}")
simple_d = np.expm1(path)
simple_a = np.expm1(path.reshape(-1, 252).sum(axis=1))
print(f"   simple, daily mean x 252: {simple_d.mean() * 252:.3%}   annual mean: {simple_a.mean():.3%}")

# 5.
for n_obs in [3, 5, 10, 30]:
    days = n_obs * trading_days
    R5 = daily_log_returns(n_paths, days, rng).reshape(n_paths, n_obs, trading_days).sum(axis=2)
    mu_hat = R5.mean(axis=1)
    se_hat = R5.std(axis=1, ddof=1) / np.sqrt(n_obs)
    cz = np.mean(np.abs(mu_hat - m_log) <= 1.96 * se_hat)
    ct = np.mean(np.abs(mu_hat - m_log) <= stats.t.ppf(0.975, n_obs - 1) * se_hat)
    exact_z = 2 * stats.t.cdf(1.96, n_obs - 1) - 1
    print(f"5. n = {n_obs:2d}: coverage z {cz:.1%} (exact {exact_z:.1%}), t {ct:.1%}")

In [ ]:
# 6.
phi, months = 0.2, 120
sig_m = 0.16 / np.sqrt(12)
eps_sd = sig_m * np.sqrt(1 - phi**2)          # innovation sd so that Var(r) = sig_m^2
mu_m = m_log / 12
r = np.empty((n_paths, months))
r[:, 0] = mu_m + sig_m * rng.standard_normal(n_paths)
for t in range(1, months):
    r[:, t] = mu_m + phi * (r[:, t - 1] - mu_m) + eps_sd * rng.standard_normal(n_paths)
emp = (r.mean(axis=1) * 12).std()
print(f"6. empirical SE {emp:.2%},  sigma/sqrt(T) {0.16 / np.sqrt(10):.2%},  "
      f"long-run formula {0.16 / np.sqrt(10) * np.sqrt((1 + phi) / (1 - phi)):.2%}")

# 7.
ir = 0.5
print(f"7. years for t = 2: {years_for_t(ir):.0f};  P(3-year active return < 0) = {stats.norm.cdf(-ir * np.sqrt(3)):.1%}")

**1.** SE $= 17\%/\sqrt5 = 7.6\%$, interval $7\% \pm 14.9\%$, about $[-7.9\%,\ 21.9\%]$.

**2.** $(11.1/2)^2 = 31$ years.

**3.** The standard error is $15\%/\sqrt3 = 8.7\%$ and the 95% interval roughly $[-5\%,\ 29\%]$. Three years cannot
distinguish this fund from one with the market's expected return, or even from one that loses money in
expectation. The 5-point lead is well inside one standard error. An investor should treat the 12% as
weak evidence and look for reasons beyond the track record (process, fees, risk taken).

**4.** For log returns the weekly and daily estimates agree exactly (a 252-day year has $252/5 = 50.4$ five-day weeks,
hence the factor). For simple returns they differ, here −3.76% against −3.56%. Both estimate roughly the 6.5% expected
simple return, not the 5.22% log mean. Annual simple returns include compounding within the year, so their
expectation is slightly higher ($e^{0.065} - 1 = 6.7\%$ against 6.5%); in one history the sample noise is much larger
than that gap. Multiplying a mean daily simple return by 252 is an approximation.

**5.** The z-interval covers about 81% at $n = 3$, 88% at $n = 5$, 92% at $n = 10$ and 94% at $n = 30$, matching the exact
values $P(|t_{n-1}| \le 1.96)$ within simulation noise (about ±0.6 points with 5,000 histories). The t-interval
stays near 95% throughout, because the returns are normal. With fewer than about 30 annual observations, use $t$.

**6.** With $\phi = 0.2$ the long-run factor is $\sqrt{1.2/0.8} = 1.22$: the standard error is about 22% larger than
$\sigma/\sqrt T$. Positive autocorrelation (smoothed returns, common in illiquid assets and private markets) makes
the mean **less** precise than the naive formula says, while making the reported volatility look lower.

**7.** $(2/0.5)^2 = 16$ years. At three years the active return is $N(0.5\,\mathrm{TE},\ \mathrm{TE}^2/3)$ per year, so
$P(\text{negative}) = \Phi(-0.5\sqrt3) = 19.3\%$: about one in five genuinely skilled managers would be fired.

## 9. Conclusion

- The standard error of an annualized mean return is $\sigma/\sqrt T$: it depends on the number of **years**, not on the
  number of observations. For log returns the estimate is exactly $(\ln P_T - \ln P_0)/T$.
- Ten years of equity data give a standard error of about 5% per year; a 1% standard error needs about 250 years.
- Volatility is different: its standard error is about $\sigma/\sqrt{2n}$, so daily data estimates it far better.
- The 95% interval from daily data covers well even with fat tails; with a handful of annual returns,
  use the t distribution.
- Telling a Sharpe ratio of 0.375 from zero at $t = 2$ needs about 28 years.

Next: [06/02 Confidence interval for a Sharpe ratio](./02_confidence_interval_for_a_sharpe_ratio.ipynb).